# Train — Credit Default
## Experimento de entrenamiento reproducible

Notebook equivalente al flujo `infraestructure/flows/entrenamiento_flow.py`: carga datos, separa train/test, construye modelos, evalúa y registra resultados en MLflow.

## 0. Librerías

In [ ]:
import os
from pathlib import Path
import pandas as pd
import yaml
import mlflow
import skops
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, average_precision_score)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns",None)

c:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\mlops-delinquency-prediction-h6cz1PHb-py3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración del proyecto

In [2]:
RAIZ=Path.cwd()
if RAIZ.name=="notebooks": RAIZ=RAIZ.parent
CONFIG=RAIZ/"configs"/"config.yaml"
with open(CONFIG,encoding="utf-8") as f:
    cfg=yaml.safe_load(f)
cfg

{'datos': {'ruta': 'data/default of credit card clients.csv',
  'target': 'default_payment_next_month',
  'id': 'ID'},
 'division': {'test_size': 0.2, 'random_state': 42},
 'features': [{'nombre': 'all', 'columnas': 'ALL'},
  {'nombre': 'custom',
   'columnas': ['LIMIT_BAL',
    'AGE',
    'EDUCATION',
    'MARRIAGE',
    'PAY_0',
    'PAY_2',
    'PAY_3',
    'PAY_4',
    'PAY_5',
    'PAY_6',
    'BILL_AMT1',
    'BILL_AMT2',
    'BILL_AMT3',
    'BILL_AMT4',
    'BILL_AMT5',
    'BILL_AMT6',
    'PAY_AMT1',
    'PAY_AMT2',
    'PAY_AMT3',
    'PAY_AMT4',
    'PAY_AMT5',
    'PAY_AMT6']}],
 'candidatos': [{'nombre_run': 'credit-logreg-c1',
   'tipo': 'logistic_regression',
   'parametros': {'C': 1.0, 'max_iter': 1000, 'class_weight': 'balanced'}},
  {'nombre_run': 'credit-logreg-c01',
   'tipo': 'logistic_regression',
   'parametros': {'C': 0.1, 'max_iter': 1000, 'class_weight': 'balanced'}},
  {'nombre_run': 'credit-random-forest',
   'tipo': 'random_forest',
   'parametros': {'n_es

## 2. Carga y preparación del dataset

In [5]:
ruta=(RAIZ/cfg["datos"]["ruta"]).resolve()
df=pd.read_csv(ruta,sep=";",encoding="utf-8",low_memory=False)
df.columns=df.columns.str.strip().str.replace(" ","_")
TARGET = "default_payment_next_month"
if TARGET not in df.columns:
    candidatos=[c for c in df.columns if "default" in c.lower() and "next" in c.lower()]
    if candidatos: df=df.rename(columns={candidatos[0]:TARGET})
assert TARGET in df.columns, f"No existe target {TARGET}"
ID_COL = "ID"
# ID y target quedan fuera de X
FEATURES=[c for c in df.columns if c not in [TARGET,ID_COL]]
X=df[FEATURES].copy(); y=df[TARGET].astype(int)
print("Ruta:",ruta," | Shape:",df.shape)
print("Features:",len(FEATURES),FEATURES)
print("Tasa default:",y.mean())
display(df.head())

Ruta: C:\Users\ASUS\Actividad-2-MLOps\data\default of credit card clients.csv  | Shape: (30000, 25)
Features: 23 ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
Tasa default: 0.2212


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_payment_next_month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 3. División estratificada train/test

In [6]:
train_df,test_df=train_test_split(
    df,test_size=cfg["division"]["test_size"],
    random_state=cfg["division"]["random_state"],
    stratify=y
)
X_train=train_df[FEATURES]; y_train=train_df[TARGET].astype(int)
X_test=test_df[FEATURES]; y_test=test_df[TARGET].astype(int)
print("Train:",X_train.shape,"Test:",X_test.shape)
print("Default train/test:",y_train.mean(),y_test.mean())

Train: (24000, 23) Test: (6000, 23)
Default train/test: 0.22120833333333334 0.22116666666666668


## 4. Preprocesamiento y candidatos

In [7]:
# Tipos de variables coherentes con domain/modelo.py
categoricas=[c for c in ["SEX","EDUCATION","MARRIAGE"] if c in FEATURES]
numericas=[c for c in FEATURES if c not in categoricas]
preprocesador=ColumnTransformer([
    ("num",StandardScaler(),numericas),
    ("cat",OneHotEncoder(handle_unknown="ignore"),categoricas)
])

candidatos={
    "logreg_c1": LogisticRegression(C=1, max_iter=1000, class_weight="balanced",random_state=42),
    "logreg_c01": LogisticRegression(C=.1, max_iter=1000, class_weight="balanced",random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200,max_depth=12,
        min_samples_leaf=100,class_weight="balanced",random_state=42,n_jobs=-1)
}
print("Numéricas:",len(numericas),"Categóricas:",categoricas)

Numéricas: 20 Categóricas: ['SEX', 'EDUCATION', 'MARRIAGE']


## 5. Entrenamiento y evaluación

In [8]:
resultados=[]
umbral=cfg["metricas"]["umbral"]
for nombre,estimador in candidatos.items():
    pipeline=Pipeline([("preprocesado",preprocesador),("modelo",estimador)])
    pipeline.fit(X_train,y_train)
    proba=pipeline.predict_proba(X_test)[:,1]
    pred=(proba>=umbral).astype(int)
    metricas={
        "auc":float(roc_auc_score(y_test,proba)),
        "auc_pr":float(average_precision_score(y_test,proba)),
        "accuracy":float(accuracy_score(y_test,pred)),
        "precision":float(precision_score(y_test,pred,zero_division=0)),
        "recall":float(recall_score(y_test,pred,zero_division=0)),
        "f1":float(f1_score(y_test,pred,zero_division=0))
    }
    resultados.append({"nombre":nombre,"pipeline":pipeline,"metricas":metricas})
    print(nombre,metricas)
tabla=pd.DataFrame([{ "modelo":r["nombre"],**r["metricas"]} for r in resultados])
display(tabla.sort_values("auc",ascending=False))

logreg_c1 {'auc': 0.7105606434759414, 'auc_pr': 0.49102029693866645, 'accuracy': 0.6781666666666667, 'precision': 0.36765994741454866, 'recall': 0.6322532027128862, 'f1': 0.4649487392629537}
logreg_c01 {'auc': 0.7101479728259843, 'auc_pr': 0.49108927689709564, 'accuracy': 0.6791666666666667, 'precision': 0.36862917398945516, 'recall': 0.6322532027128862, 'f1': 0.46572300860394117}
random_forest {'auc': 0.7736583728842968, 'auc_pr': 0.5430979363695378, 'accuracy': 0.7595, 'precision': 0.46708286038592506, 'recall': 0.6201959306706858, 'f1': 0.5328585302686953}


,modelo,auc,auc_pr,accuracy,precision,recall,f1
2,random_forest,0.773658,0.543098,0.759500,0.467083,0.620196,0.532859
0,logreg_c1,0.710561,0.491020,0.678167,0.367660,0.632253,0.464949
1,logreg_c01,0.710148,0.491089,0.679167,0.368629,0.632253,0.465723


## 6. Registro en MLflow

In [10]:
tracking_uri=os.environ.get("MLFLOW_TRACKING_URI",cfg["mlflow"]["tracking_uri"])
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(cfg["mlflow"]["experimento"])

runs=[]
for r in resultados:
    with mlflow.start_run(run_name=f"notebook-{r['nombre']}") as run:
        mlflow.set_tag("fase","entrenamiento-notebook")
        mlflow.set_tag("dataset","credit-default")
        mlflow.log_params({"modelo":r["nombre"],"semilla":cfg["division"]["random_state"],
                            "n_features":len(FEATURES),"umbral":umbral})
        mlflow.log_metrics(r["metricas"])
        signature=infer_signature(X_test,r["pipeline"].predict(X_test))

        info = mlflow.sklearn.log_model(
            sk_model=r["pipeline"],
            name="model",
            signature=signature,
            input_example=X_test.head(3),
            skops_trusted_types=[
                "sklearn.tree._tree.Tree"
            ]
        )

        runs.append({"modelo":r["nombre"],"run_id":run.info.run_id,
                     "model_uri":info.model_uri,"auc":r["metricas"]["auc"]})
display(pd.DataFrame(runs))
print("Tracking URI:",tracking_uri)

c:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\mlops-delinquency-prediction-h6cz1PHb-py3.13\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run notebook-logreg_c1 at: http://localhost:5000/#/experiments/2/runs/45ec8c6a89764d61acff0dc9a8471692
🧪 View experiment at: http://localhost:5000/#/experiments/2


c:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\mlops-delinquency-prediction-h6cz1PHb-py3.13\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run notebook-logreg_c01 at: http://localhost:5000/#/experiments/2/runs/37700b3c5c3040519cf3157ab6aeb74c
🧪 View experiment at: http://localhost:5000/#/experiments/2


c:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\mlops-delinquency-prediction-h6cz1PHb-py3.13\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run notebook-random_forest at: http://localhost:5000/#/experiments/2/runs/b2a06e8eeb1447a4b270aaf2cfcdf016
🧪 View experiment at: http://localhost:5000/#/experiments/2


,modelo,run_id,model_uri,auc
0,logreg_c1,45ec8c6a89764d61acff0dc9a8471692,models:/m-914901a2915c4687b1a5aba89e8b6593,0.710561
1,logreg_c01,37700b3c5c3040519cf3157ab6aeb74c,models:/m-62c40cbc3ded47f28b02ae4556a4e78f,0.710148
2,random_forest,b2a06e8eeb1447a4b270aaf2cfcdf016,models:/m-0c58dedc4a4b4b24b459a6b48fb2c36b,0.773658


Tracking URI: http://localhost:5000


## 7. Selección del mejor candidato

In [11]:
mejor=max(resultados,key=lambda r:r["metricas"]["auc"])
print("Mejor modelo:",mejor["nombre"])
print("Métricas:",mejor["metricas"])
print("El notebook registra runs; la promoción a alias champion se realiza en entrenamiento_flow.py.")

Mejor modelo: random_forest
Métricas: {'auc': 0.7736583728842968, 'auc_pr': 0.5430979363695378, 'accuracy': 0.7595, 'precision': 0.46708286038592506, 'recall': 0.6201959306706858, 'f1': 0.5328585302686953}
El notebook registra runs; la promoción a alias champion se realiza en entrenamiento_flow.py.


## 8. Diferencia respecto al pipeline Prefect

- Este notebook permite experimentar, visualizar métricas y explicar el proceso.

